<a href="https://colab.research.google.com/github/Jahnavi-verma/AI/blob/main/1BF24CS126_WEEK8_ForwardFOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re

class Clause:
    def __init__(self, premises, conclusion):
        # premises: list of literals, conclusion: single literal
        self.premises = premises
        self.conclusion = conclusion

    def __repr__(self):
        if not self.premises:
            return f"{self.conclusion}"
        return f"{' ^ '.join(self.premises)} => {self.conclusion}"

def parse_literal(literal):
    # Extract predicate and arguments: e.g., Sells(Robert, T1, A) -> ('Sells', ('Robert', 'T1', 'A'))
    match = re.match(r"(\w+)\((.*)\)", literal.strip())
    if match:
        pred = match.group(1)
        args = tuple(arg.strip() for arg in match.group(2).split(','))
        return pred, args
    return literal.strip(), ()

def substitute(theta, literal):
    if theta is None:
        theta = {}
    pred, args = parse_literal(literal)
    if not args:
        return literal
    new_args = tuple(theta.get(arg, arg) for arg in args)
    return f"{pred}({', '.join(new_args)})"

def unify_literal(lit1, lit2, theta=None):
    if theta is None:
        theta = {}
    p1, args1 = parse_literal(lit1)
    p2, args2 = parse_literal(lit2)
    if p1 != p2 or len(args1) != len(args2):
        return None

    new_theta = theta.copy()
    for a1, a2 in zip(args1, args2):
        val1 = new_theta.get(a1, a1)
        val2 = new_theta.get(a2, a2)
        if val1 != val2:
            # Simple matching: if one is a variable (lowercase single char or variable naming convention)
            is_var1 = len(val1) == 1 and val1.islower()
            is_var2 = len(val2) == 1 and val2.islower()
            if is_var1:
                new_theta[val1] = val2
            elif is_var2:
                new_theta[val2] = val1
            else:
                return None
    return new_theta

def fol_fc_ask(KB, alpha):
    """First-Order Logic Forward Chaining Algorithm"""
    # Separate facts and rules
    facts = set()
    rules = []
    for clause in KB:
        if not clause.premises:
            facts.add(clause.conclusion)
        else:
            rules.append(clause)

    print("\n--- Starting Forward Chaining Deduction Loop ---")
    step = 1
    while True:
        new_sentences = set()
        for rule in rules:
            # Find substitutions that satisfy all premises using current facts
            # We implement a simple matching for rule premises p_1 ^ ... ^ p_n against facts
            def match_premises(premises, theta, index=0):
                if theta is None:
                    return
                if index == len(premises):
                    yield theta
                    return

                premise = premises[index]
                sub_premise = substitute(theta, premise)
                for fact in facts:
                    phi = unify_literal(sub_premise, fact, theta)
                    if phi is not None:
                        yield from match_premises(premises, phi, index + 1)

            for theta in match_premises(rule.premises, {}):
                q_prime = substitute(theta, rule.conclusion)

                # If q' is not already in KB (facts) or new
                if q_prime not in facts and q_prime not in new_sentences:
                    print(f"[Step {step}] Inferred: {q_prime} from rule [{rule}] with theta={theta}")
                    new_sentences.add(q_prime)
                    step += 1

                    # Check if it unifies with query alpha
                    phi = unify_literal(q_prime, alpha)
                    if phi is not None:
                        print(f"\nSUCCESS: Query '{alpha}' proven with substitution: {phi}")
                        return phi

        if not new_sentences:
            print("\nFAILED: No new sentences could be derived. Query cannot be proven.")
            return False

        facts.update(new_sentences)

In [4]:
# Define the initial Knowledge Base based on your FOL representation
# Variables are lower case single characters (p, q, r, x)
initial_kb = [
    # Rules
    Clause(["American(p)", "Weapon(q)", "Sells(p, q, r)", "Hostile(r)"], "Criminal(p)"),
    Clause(["Missile(x)", "Owns(A, x)"], "Sells(Robert, x, A)"),
    Clause(["Missile(x)"], "Weapon(x)"),
    Clause(["Enemy(x, America)"], "Hostile(x)"),

    # Facts
    Clause([], "Owns(A, T1)"),
    Clause([], "Missile(T1)"),
    Clause([], "American(Robert)"),
    Clause([], "Enemy(A, America)")
]

# Terminal-like interactive prompt
print("=== FOL Forward Chaining System ===")
query_input = input("Enter the query to prove (default: Criminal(Robert)): ").strip()
if not query_input:
    query_input = "Criminal(Robert)"

# Run FC
result = fol_fc_ask(initial_kb, query_input)

=== FOL Forward Chaining System ===
Enter the query to prove (default: Criminal(Robert)): Criminal(Robert)

--- Starting Forward Chaining Deduction Loop ---
[Step 1] Inferred: Sells(Robert, T1, A) from rule [Missile(x) ^ Owns(A, x) => Sells(Robert, x, A)] with theta={'x': 'T1'}
[Step 2] Inferred: Weapon(T1) from rule [Missile(x) => Weapon(x)] with theta={'x': 'T1'}
[Step 3] Inferred: Hostile(A) from rule [Enemy(x, America) => Hostile(x)] with theta={'x': 'A'}
[Step 4] Inferred: Criminal(Robert) from rule [American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) => Criminal(p)] with theta={'p': 'Robert', 'q': 'T1', 'r': 'A'}

SUCCESS: Query 'Criminal(Robert)' proven with substitution: {}
